# ✅ 1. Criar parâmetros no SSM Parameter Store: 
ATENÇÃO para: ALTERAR O CAMPO s3-bucket-name, APÓS CRIAR a infra pelo terraform anote o nome da bucket criada e insira no parâmetro.

In [ ]:
# Configurações de Infraestrutura AWS
aws ssm put-parameter --name "/chatbot-juridico/s3-bucket-name" --type "String" --value "INSERIR-NOME-DA-BUCKET-CRIADA-PELO-TERRAFORM" --overwrite
aws ssm put-parameter --name "/chatbot-juridico/log-group" --type "String" --value "/aws/chatbot-consultor-juridico" --overwrite

# Configurações do Bedrock
aws ssm put-parameter --name "/chatbot-juridico/bedrock-model-id" --type "String" --value "amazon.titan-embed-text-v2:0" --overwrite
aws ssm put-parameter --name "/chatbot-juridico/bedrock-batch-size" --type "String" --value "48" --overwrite
aws ssm put-parameter --name "/chatbot-juridico/bedrock-max-retries" --type "String" --value "3" --overwrite
aws ssm put-parameter --name "/chatbot-juridico/bedrock-retry-multiplier" --type "String" --value "1" --overwrite
aws ssm put-parameter --name "/chatbot-juridico/bedrock-min-retry-delay" --type "String" --value "2" --overwrite
aws ssm put-parameter --name "/chatbot-juridico/bedrock-max-retry-delay" --type "String" --value "10" --overwrite
aws ssm put-parameter --name "/chatbot-juridico/bedrock-batch-delay" --type "String" --value "0.15" --overwrite
aws ssm put-parameter --name "/chatbot-juridico/bedrock-text-truncate" --type "String" --value "7500" --overwrite

# Configurações de PDF Processing
aws ssm put-parameter --name "/chatbot-juridico/chunk-size" --type "String" --value "1200" --overwrite
aws ssm put-parameter --name "/chatbot-juridico/chunk-overlap" --type "String" --value "300" --overwrite
aws ssm put-parameter --name "/chatbot-juridico/max-tokens" --type "String" --value "8000" --overwrite
aws ssm put-parameter --name "/chatbot-juridico/pdf-prefix" --type "String" --value "juridicos/" --overwrite
aws ssm put-parameter --name "/chatbot-juridico/min-valid-chunk-lines" --type "String" --value "3" --overwrite

# ✅ 2. Anexar permissões a uma IAM Role (Lambda ou EC2)
Aqui vai uma policy customizada que pode ser anexada a uma IAM role que o script utilizará:
você precisa de uma Policy IAM (Identity and Access Management) na AWS para permitir que a entidade que está rodando sua aplicação (neste caso, provavelmente um Role IAM associado à sua instância EC2) acesse e leia esses parâmetros no SSM Parameter Store.

In [ ]:
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "ssm:GetParameter",
                "ssm:GetParameters"
            ],
            "Resource": "arn:aws:ssm:<region>:<account-id>:parameter/chatbot-juridico/*"
        },
        {
            "Effect": "Allow",
            "Action": [
                "kms:Decrypt"
            ],
            "Resource": "arn:aws:kms:<region>:<account-id>:key/*",
            "Condition": {
                "StringLike": {
                    "kms:ViaService": "ssm.<region>.amazonaws.com"
                }
            }
        }
    ]
}

### Para criar a policy via CLI:
Salve o JSON acima como chatbot_parameters_policy.json

Rode:

In [ ]:
aws iam create-policy \
  --policy-name ChatbotParametersPolicy \
  --policy-document file://chatbot_parameters_policy.json


### Depois, anexe essa policy à role usada pelo Lambda/EC2:


In [ ]:
aws iam attach-role-policy \
  --role-name MinhaRoleLambdaOuEC2 \
  --policy-arn arn:aws:iam::<ACCOUNT_ID>:policy/ChatbotParametersPolicy

# (Substitua <ACCOUNT_ID> pelo ID da conta AWS)